# bootstrap


# Logistic

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pandas as pd
import joblib



review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv")


print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})

vectorizer = TfidfVectorizer()

review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# Array representing the labels. Unlabeled samples should have the label -1.
review_y = review_data["recommend"].fillna(-1)


# print(vectorizer.get_feature_names_out())  # Shows vocabulary
# review_X_train, review_X_test, review_y_train, review_y_test = train_test_split(review_X, review_y, test_size=0.2, random_state=2023)

# print(f"count for missing data {len(review_y_train[review_y_train==-1])}")
# print(f"count for labeled data {len(review_y_train[review_y_train!=-1])}")
# print(f"count for missing data_test {len(review_y_test [review_y_test ==-1])}")
# print(f"count for labeled data_Test {len(review_y_test [review_y_test !=-1])}")

logit_review_model = LogisticRegression()





# Bert

In [1]:

from tqdm import tqdm
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset
from multiprocessing import Pool, cpu_count
# Train


    
# this needed to deal with memory error, I kept running out of memroy
# it may also be issue with the enviroment since I tested the same code in jupyter note book and it als crashed so likly not an IDE issue



def predictions(model, data, batch_size=64):
    """ Get predictions on data for a classification model m returning predictions and true labels"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
 
    all_predictions = []

    
    if (data['input_ids']) % batch_size != 0:
        num_batches = len(data['input_ids']) // batch_size + 1
    else:
        num_batches = len(data['input_ids']) // batch_size

    for i in tqdm(range(num_batches),desc= "Prediction Progress"):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(data['input_ids'])) # prevent going out of bount input_ids has same size as attention mask should
        input_ids_batch = torch.tensor(data['input_ids'][start_idx:end_idx])
        attention_mask_batch = torch.tensor(data['attention_mask'][start_idx:end_idx])
        with torch.no_grad():  # Disable gradient computation for inference
            predictions = model(input_ids_batch, attention_mask=attention_mask_batch)
            
        all_predictions.append(predictions)
        # print(predictions)
        # print("--------")
    # print(all_predictions)
    
    
    #Concatenate all batch predictions into one tensor since we currently have a list of lists

    all_confidences = torch.cat(all_confidences, dim=0)
    
    return all_predictions

def bootstrap_sample(dataset, seed=None):
    n = len(dataset)
    rng = random.Random(seed)  # Local random generator
    indices = [rng.randint(0,n-1) for _ in range(n)]
    return dataset.select(indices)

def train_single_boot_model(args):
    
    seed, model, train_data, num_epochs, tokenizer = args

    bootstrapped_train_data = bootstrap_sample(train_data, seed)
    data_collator = DataCollatorWithPadding(tokenizer)


    training_args = TrainingArguments(
        output_dir="test_trainer",        # Directory to save logs and model checkpoints
        num_train_epochs=num_epochs,      # Number of training epochs
    )

    # Initialize the Trainer with the model, training arguments, and datasets
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=bootstrapped_train_data ,    # Training dataset     # Evaluation dataset (for validation)
        data_collator=data_collator,
    )

    # Train the model
    trainer.train()
    return model




def train_models_in_parallel(train_data, num_epochs=1, tokenizer=None, num_models=None):
    """Train multiple models in parallel using bootstrapped datasets."""
    if num_models is None:
        num_models = cpu_count()

    seeds = [2025 + i for i in range(num_models)]

    args_list = [(train_data, num_epochs, tokenizer, seed) for seed in seeds]

    with Pool(min(cpu_count(),num_models)) as pool:
        models = pool.map(train_single_boot_model, args_list)

    return models


def parallel_predictions(models, data, batch_size=64):
    """ Get predictions for all models in parallel."""
    predctions_list = [(model, data, batch_size) for model in models]
    with Pool(min(cpu_count(),len(models))) as pool:
        results = pool.starmap(get_predictions_for_model, predctions_list) # unpacks the tuple first
    
    # Now you have a list of predictions for each model
    combined_predictions =  combined_binary_pred(results)  # Combine predictions of shape [num_models, num_samples, num_classes]
    return combined_predictions

def combined_binary_pred(predictions):
    predictions_tensor = torch.tensor(predictions) 
    
    # Average across models (axis 0), then apply threshold of 0.5 to get final predictions
    averaged_predictions = torch.mean(predictions_tensor, dim=0)
    
    # Apply threshold of 0.5 to get binary output (1 if > 0.5, else 0) 
    combined_predictions = (averaged_predictions => 0.5).int()  # Convert boolean to int produces a bollen list then 
    
    return combined_predictions
    
    


/opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <0B7EB158-53DC-3403-8A49-22178CAB4612> /opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/image.so
  Reason: tried: '/opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/cosc410/lib/python3.10/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/cosc410/lib/python3.10/lib-dynload/../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/cosc410/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `li

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch
from datasets import Dataset
from multiprocessing import Pool, cpu_count


review_data = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/unlabeled_reviews.csv",nrows=5000)

print(review_data["review_comment"].isna().sum())
review_data["review_comment"] = review_data["review_comment"].fillna("") 
review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})



# vectorizer = TfidfVectorizer()

# review_X = vectorizer.fit_transform(review_data["review_comment"])  # Convert text to TF-IDF features

# # Array representing the labels. Unlabeled samples should have the label -1.

review_data["recommend"] = review_data["recommend"].fillna(-1)


print("wow")
print(review_data[review_data["recommend"]==-1])


# for some reason the tyepe matter https://discuss.huggingface.co/t/valueerror-target-size-torch-size-8-must-be-the-same-as-input-size-torch-size-8-8/12133/9
train_df = pd.DataFrame({"text": review_data["review_comment"], "label": review_data["recommend"].astype(int)})



# Now we are working with huggingface daset 

train_dataset = Dataset.from_pandas(train_df)

print(sum(label == -1 for label in train_dataset["label"]))
print(train_dataset["label"])
unlabeled_mask =  train_dataset["label"] == -1
print(unlabeled_mask)
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Bert model accepts max 512
# Tokenize datasets
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=512)

#------------------------------------


#------------------------------------

# In Hugging Face models like BERT, the features come from the tokenized input data that is fed into the model.
# the map applie the tokenize function to each batch of the function 
train_dataset = train_dataset.map(tokenize, batched=True)

# print(train_dataset)
# Load model
# define model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)






models = train_models_in_parallel(train_dataset, 1, tokenizer=5)
train_dataset = parallel_predictions(models, train_dataset, batch_size=64)


# Save the trained model and tokenizer
model.save_pretrained("bootstrap_bert_model")
tokenizer.save_pretrained("bootstrap_tokenizer")

# Save the final labeled dataset
train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
train_dataset_df = train_dataset_df[["label","text"]]
train_dataset_df.to_csv("final_bootstrap_labeled_reviews.csv", index=False)









#---------Evaluate-----------



train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
train_dataset_df = train_dataset_df[["label","text"]]
train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)
review_data_test = pd.read_csv("/Users/apple/Documents/GitHub/Steam-Market-Data-ML/cleaned_reviews.csv",nrows=5000)



print(review_data_test ["review_comment"].isna().sum())
review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})
review_data_test ["recommend"] = review_data_test ["recommend"].fillna(-1)



review_y_test = review_data_test["recommend"]

train_dataset_frame =  train_dataset.to_pandas()
# only the unlabled data
mask = review_data["recommend"] == -1 
review_y_test =  review_y_test[mask]

final_data_set = train_dataset_frame[mask]
print(review_y_test)
print(final_data_set)

# print(review_X_train.shape[0])
# This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
# It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.

accuracy = accuracy_score(review_y_test, final_data_set["label"] )
precision = precision_score(review_y_test, final_data_set["label"]  ,average='macro')
recall = recall_score(review_y_test,final_data_set["label"]  ,average='macro')


print("\nEvaluation Results on Test Data:")
print(f"Accuracy:  {accuracy}")
print(f"Precision: {precision}")
print(f"Recall:    {recall}")

with open('evaluation_results_selftrain_noboot_bert_models.txt', 'w') as f:
    f.write(f"Accuracy after fine-tuning: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")


    



In [ ]:
  

def main():
    torch.cuda.empty_cache()

    # We are still in datafre
    set_start_method('spawn', force=True)
    review_data = pd.read_csv("Machine_Learning_final/opposite_data_reviews.csv",nrows = 10)
    # review_data = pd.read_csv("Machine_Learning_final/opposite_data_reviews.csv")
    print(review_data.shape)
    
    print(review_data["review_comment"].isna().sum())
    review_data["review_comment"] = review_data["review_comment"].fillna("") 
    review_data["recommend"] = review_data["recommend"].map({"Recommended": 1, "Not Recommended": 0})
    
    review_data["recommend"] = review_data["recommend"].fillna(-1)
    
    
    # print(torch.cuda.get_device_name(0))

    # # Run nvidia-smi and capture the output
    # try:
    #     result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
    #     print("nvidia-smi output:\n", result.stdout)
    # except subprocess.CalledProcessError as e:
    #     print("Error running nvidia-smi:", e.stderr)
    # print(review_data[review_data["recommend"]==-1])
    
    
    # for some reason the tyepe matter https://discuss.huggingface.co/t/valueerror-target-size-torch-size-8-must-be-the-same-as-input-size-torch-size-8-8/12133/9
    train_df = pd.DataFrame({
    "text": review_data["review_comment"], 
    "label": review_data["recommend"].apply(lambda x: int(x) if x in [0, 1, -1] else -1)
    })
    
    
    
    # Now we are working with huggingface daset 
    
    train_dataset = Dataset.from_pandas(train_df)
    
    # print(sum(label == -1 for label in train_dataset["label"]))
    # print(train_dataset["label"])
    unlabeled_mask =  train_dataset["label"] == -1
    print(unlabeled_mask)
    # Load tokenizer
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    
    # Bert model accepts max 512
    # Tokenize datasets
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=512)
    
    train_dataset = train_dataset.map(tokenize, batched=True)
    
    # print(train_dataset)
    # Load model
    # define model
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    
    
    
    models = train_models_in_parallel(train_dataset, model, 1, tokenizer, 1)
    train_dataset = parallel_predictions(models, train_dataset, batch_size=64)
    
    
    # Save the trained model and tokenizer
    model.save_pretrained("bootstrap_bert_model")
    tokenizer.save_pretrained("bootstrap_tokenizer")
    
    # Save the final labeled dataset
    train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
    train_dataset_df = train_dataset_df[["label","text"]]
    train_dataset_df.to_csv("final_bootstrap_labeled_reviews.csv", index=False)
    


    
    #---------Evaluate-----------
    
    
    
    train_dataset_df = train_dataset.to_pandas()  # Convert dataset back to pandas DataFrame
    train_dataset_df = train_dataset_df[["label","text"]]
    train_dataset_df.to_csv("final_labeled_reviews.csv", index=False)
    review_data_test = pd.read_csv("Machine_Learning_final/cleaned_reviews.csv",nrows = 10)
    # review_data_test = pd.read_csv("Machine_Learning_final/cleaned_reviews.csv")
    
    
    
    print(review_data_test ["review_comment"].isna().sum())
    review_data_test ["review_comment"] = review_data_test ["review_comment"].fillna("") 
    review_data_test ["recommend"] = review_data_test ["recommend"].map({"Recommended": 1, "Not Recommended": 0})
    review_data_test ["recommend"] = review_data_test ["recommend"].fillna(-1)
    
    
    
    review_y_test = review_data_test["recommend"]
    
    train_dataset_frame =  train_dataset.to_pandas()
    # only the unlabled data
    mask = review_data["recommend"] == -1 
    review_y_test =  review_y_test[mask]
    
    final_data_set = train_dataset_frame[mask]
    print(review_y_test)
    print(final_data_set)
    
    # print(review_X_train.shape[0])
    # This metaestimator allows a given supervised classifier to function as a semi-supervised classifier, allowing it to learn from unlabeled data. 
    # It does this by iteratively predicting pseudo-labels for the unlabeled data and adding them to the training set.
    
    accuracy = accuracy_score(review_y_test, final_data_set["label"] )
    precision = precision_score(review_y_test, final_data_set["label"]  ,average='macro')
    recall = recall_score(review_y_test,final_data_set["label"]  ,average='macro')
    
    
    print("\nEvaluation Results on Test Data:")
    print(f"Accuracy:  {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall:    {recall}")
    
    with open('evaluation_results_selftrain_noboot_bert_models_midway.txt', 'w') as f:
        f.write(f"Accuracy after fine-tuning: {accuracy:.4f}\n")
        f.write(f"Precision: {precision:.4f}\n")
        f.write(f"Recall: {recall:.4f}\n")
        
if __name__ == '__main__':  
    main()
